# Fault_Tolerant_Sim — end-to-end demo

Compile a logical `{H, S, T, CNOT}` circuit into one `[[6,2,2]]` circuit (one
code patch per logical qubit; `H/S/CNOT` transversal; each `T` distilled in the
H6 `[[6,2,2]]` factory and gate-teleported), then simulate it under
circuit-level depolarising noise with **clifft** (real `T` gates) and score by
post-selection.

- **acceptance** = post-selection yield (`[[6,2,2]]` is `d=2` → detection only)
- **LER** = logical error rate on the round-trip `lc ; lc†` (deterministic output)
- **noiseless_error_floor** = residual coherent error of the T gadget at `p=0`
  (bare `[[6,2,2]]` has no exact transversal `T`; a `T` conjugated by `H`
  carries ~7%). `logical_error_rate_net` subtracts it.
- **unitary_check** = compiled noiseless output distribution vs a dense
  state-vector reference.

## Step 0 — validate the package

Run the `ftsim` test suite (compile / factory / gadget / pipeline / clifft
smoke). If anything fails, stop — the sweep below would only produce garbage.

In [1]:
import subprocess, sys
from pathlib import Path

ROOT = Path.cwd().parent
p = subprocess.run([sys.executable, '-m', 'pytest', str(ROOT / 'tests'), '-q', '--no-header'],
                   cwd=ROOT, capture_output=True, text=True)
print(p.stdout[-2000:])
if p.returncode != 0:
    print(p.stderr[-2000:])
    raise RuntimeError(f'ftsim tests failed (exit {p.returncode})')
print('OK — ftsim validated.')

.................................                                        [100%]

OK — ftsim validated.


In [ ]:
import pandas as pd
from ftsim import LogicalCircuit, run_pipeline


## Configure the logical circuit

A `LogicalCircuit` is a list of `('H', q) | ('S', q) | ('T', q) | ('CNOT', c, t)`
(plus `S_DAG`, `T_DAG`, `X`). Keep the T-count small — clifft cost grows with
simultaneously-live magic, and acceptance falls ~2× per `T`.

In [3]:
# A small T-using circuit. (Avoid a T immediately conjugated by H on the same
# qubit -- bare [[6,2,2]] transversal T is not exact there; it shows up as the
# noiseless_error_floor.)
lc = LogicalCircuit([
    ('T', 0),
    ('CNOT', 0, 1),
    ('S', 0),
    ('T', 1),
])
lc

LogicalCircuit(n=2, t=2, [T(0,), CNOT(0, 1), S(0,), T(1,)])

## Sweep the physical error rate

In [4]:
import numpy as np, clifft
from ftsim.sim.noise import add_noise

def bare_ler(lc, p, shots, seed=0, mode='full'):
    """LER of the *unencoded* round-trip lc ; lc.inverse(): one physical qubit
    per logical qubit, physical T gates, no [[6,2,2]] code, no post-selection.
    Ideal output = all-zeros."""
    n = lc.n_qubits
    nl = chr(10)
    gates = list(lc.gates) + list(lc.inverse().gates)
    q_all = ' '.join(map(str, range(n)))
    lines = [f'R {q_all}']
    for g in gates:
        lines.append(f'CX {g[1]} {g[2]}' if g[0] == 'CNOT' else f'{g[0]} {g[1]}')
    lines.append(f'M {q_all}')
    lines += [f'OBSERVABLE_INCLUDE({q}) rec[{q - n}]' for q in range(n)]
    r = clifft.sample(clifft.compile(add_noise(nl.join(lines) + nl, p, mode)),
                      shots=shots, seed=seed)
    obs = np.asarray(r.observables, dtype=np.uint8)
    return float((obs != 0).any(axis=1).mean())

P_VALUES = [0.0, 1e-3, 3e-3, 1e-2]
SHOTS = 100_000

rows = []
for p in P_VALUES:
    r = run_pipeline(lc, p=p, shots=SHOTS, check=(p == 0.0))
    rows.append({
        'p': p,
        'accept': r.post_selection_rate,
        'LER': r.logical_error_rate,
        'LER_net': r.logical_error_rate_net,
        'LER_bare': bare_ler(lc, p, SHOTS),
        'floor': r.noiseless_error_floor,
        'passed': r.passed_shots,
        'seconds': round(r.seconds, 1),
    })

df = pd.DataFrame(rows)
df

,p,accept,LER,LER_net,LER_bare,floor,passed,seconds
0,0.000,0.00946,0.000000,0.000000,0.00000,0.0,946,0.3
1,0.001,0.00471,0.002123,0.002123,0.00993,0.0,471,0.2
2,0.003,0.00146,0.006849,0.006849,0.02883,0.0,146,0.3
3,0.010,0.00001,0.000000,0.000000,0.09057,0.0,1,0.3


In [ ]:
import matplotlib.pyplot as plt

fig, (ax_acc, ax_ler) = plt.subplots(2, 1, sharex=True, figsize=(6, 6))
fig.suptitle(str(lc))

df.plot(x='p', y='accept', marker='o', ax=ax_acc, legend=False)
ax_acc.set_ylabel('acceptance')

df.plot(x='p', y=['LER', 'LER_bare'], marker='s', ax=ax_ler)
ax_ler.legend(['LER_h6', 'LER_baseline'])
ax_ler.set_ylabel('logical error rate')
ax_ler.set_xlabel('physical error rate p')

fig.tight_layout()